In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
import warnings
warnings.filterwarnings('ignore')

print("=== AJUSTE DE UMBRAL - COLDTRACK (Versión Corregida y Segura) ===\n")

# ==================== CARGAR MODELO Y COLUMNAS CORRECTAS ====================
model = joblib.load("coldtrack_rf_model_final.pkl")
feature_columns = joblib.load("feature_columns_final.pkl")

print(f"Modelo cargado: coldtrack_rf_model_final.pkl")
print(f"Número de features esperadas: {len(feature_columns)}")

# Cargar todas las features
X_full = pd.read_csv("X_features.csv")

# Usar SOLO las columnas con las que se entrenó el modelo
X_test = X_full[feature_columns].iloc[-51840:]   # últimos 51,840 registros
y_test = pd.read_csv("y_target.csv").iloc[-51840:].squeeze()

print(f"Conjunto de prueba usado: {X_test.shape[0]:,} registros × {X_test.shape[1]} features\n")

# ====================== PREDICCIONES ======================
y_proba = model.predict_proba(X_test)[:, 1]

# ====================== BUSCAR MEJOR UMBRAL ======================
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Mejor umbral según F1-score: {best_threshold:.4f}")
print(f"Precision: {precisions[best_idx]:.4f} | Recall: {recalls[best_idx]:.4f}\n")

# ====================== PROBAR UMBRALES ======================
umbrales = [0.55, 0.60, 0.65, 0.70, 0.75]

print("Evaluación con diferentes umbrales:\n")
for umbral in umbrales:
    y_pred = (y_proba >= umbral).astype(int)
    print(f"Umbral = {umbral:.2f}")
    print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Mantenimiento (1)'], digits=4))
    print("-" * 70)

# ====================== UMBRAL RECOMENDADO ======================
umbral_final = 0.68

y_pred_final = (y_proba >= umbral_final).astype(int)

print(f"\n=== RESULTADO FINAL RECOMENDADO (Umbral = {umbral_final}) ===")
print(classification_report(y_test, y_pred_final, target_names=['Normal (0)', 'Mantenimiento (1)']))

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred_final))

# Guardar umbral
joblib.dump(umbral_final, "decision_threshold.pkl")
print(f"\n✅ Umbral final guardado: {umbral_final} → decision_threshold.pkl")

print("\n¡Modelo listo para integrar con hardware y Flutter!")

=== AJUSTE DE UMBRAL - COLDTRACK (Versión Corregida y Segura) ===

Modelo cargado: coldtrack_rf_model_final.pkl
Número de features esperadas: 16
Conjunto de prueba usado: 51,840 registros × 16 features

Mejor umbral según F1-score: 0.9429
Precision: 0.9461 | Recall: 0.9813

Evaluación con diferentes umbrales:

Umbral = 0.55
                   precision    recall  f1-score   support

       Normal (0)     1.0000    0.9930    0.9965     47722
Mantenimiento (1)     0.9254    1.0000    0.9613      4118

         accuracy                         0.9936     51840
        macro avg     0.9627    0.9965    0.9789     51840
     weighted avg     0.9941    0.9936    0.9937     51840

----------------------------------------------------------------------
Umbral = 0.60
                   precision    recall  f1-score   support

       Normal (0)     1.0000    0.9930    0.9965     47722
Mantenimiento (1)     0.9254    1.0000    0.9613      4118

         accuracy                         0.9936     